# Solution 07 — the Chat class

History as an instance variable, `ask()` carrying it between turns, `save()`/`load()`
through `ModelMessagesTypeAdapter`. This is 90% of a chatbot backend.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pathlib import Path

from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessagesTypeAdapter

agent = Agent(
    MODEL,
    instructions="You are a friendly science tutor. Keep answers to two sentences.",
)


class Chat:
    def __init__(self, agent: Agent) -> None:
        self.agent = agent
        self.history = []  # list[ModelMessage]

    async def ask(self, prompt: str) -> str:
        """One turn: run the agent with the history so far, then extend it."""
        result = await self.agent.run(prompt, message_history=self.history)
        self.history = result.all_messages()
        return result.output

    def save(self, path: str | Path) -> None:
        """Serialize the whole conversation to a JSON file."""
        Path(path).write_bytes(ModelMessagesTypeAdapter.dump_json(self.history))

    @classmethod
    def load(cls, agent: Agent, path: str | Path) -> "Chat":
        """Restore a conversation from disk into a fresh Chat."""
        chat = cls(agent)
        chat.history = ModelMessagesTypeAdapter.validate_json(Path(path).read_bytes())
        return chat


chat = Chat(agent)
print("turn 1:", await chat.ask("I'm curious about black holes. Where do I start?"))
print("turn 2:", await chat.ask("Wait — what is the event horizon, exactly?"))
print("turn 3:", await chat.ask("And what happens if I cross it? Summarize what we said."))

chat.save("chat_demo.json")

revived = Chat.load(agent, "chat_demo.json")
print()
print("restored chat remembers", len(revived.history), "messages")
print("proof  :", await revived.ask("Remind me — what was the FIRST thing I asked about?"))

**Notes**

- `ask()` is the only place history mutates — one clear seam, easy to test.
- `save`/`load` are four lines because messages are plain serializable data.
- In the FastAPI phase, `save` becomes "INSERT into SQLite" and `load` becomes
  "SELECT by user id" — the class barely changes.